# Session 2 (light) — train the third arm, regularised

**~4 h, resumable.** Reads the session-1 pixel cache, writes `arm3.pt`. No
submission is produced by this notebook.

---

## 1. Kaggle settings — set all five

| setting | value | why |
|---|---|---|
| **Accelerator** | **GPU T4 × 2** | only one is used; see `USE_DP` below |
| **Internet** | **OFF** | competition rule. `pretrained=True` therefore cannot download — weights must be attached |
| **Persistence** | **Variables and Files** | so `arm3.pt` survives and a re-run resumes instead of restarting |
| **Environment** | pin to the version session 1 used | a `numpy` major change would make the cache unreadable |
| **Run** | Save Version → **Save & Run All (Commit)** | interactive sessions die at 12 h and lose the output |

## 2. Required inputs — attach exactly these two

| # | kind | what to pick | contains |
|---|---|---|---|
| 1 | **Notebook Output** | your session-1 run — the one whose output listing shows `pixels.npy` | `pixels.npy`, `masks.npy`, `labels.npy`, `pixel_index.json` |
| 2 | **Models** → search `timm` | **`resnet34`** (first choice) or `resnet50` | ImageNet weights |

Input 1 is **mandatory** — the notebook stops with a list of what is missing.
It prints every attached notebook output *and its file listing*, so if two of
your notebooks have similar names the log tells you which one you actually
attached. (That mistake cost one session.)

Input 2 is mandatory unless you set `STRICT_IMAGENET = False`. The preflight
pairs every candidate architecture against every attached weight file by
**counting matching tensors** and prints the whole table — no architecture is
ever guessed from a filename.

The competition dataset itself is **not** needed. Nothing here reads DICOM.

---

## 3. Every setting, and what decided it

### Cost — why this is "light"

| knob | Train3 | **here** | effect on one epoch |
|---|---|---|---|
| `ARCH_CANDIDATES[0]` | `resnet50` | **`resnet34`** | ~0.55× — and feature dim 512 vs 2048 shrinks the head 4× |
| `TRAIN_PX` | 160 | **128** | ×0.64 |
| `K_TRAIN` | 6 | **4** | ×0.67 |
| `BATCH_STUDIES` | 8 | **12** | fewer optimiser steps |
| `TIME_BUDGET_S` | 5.5 h | **4.0 h** | leaves headroom under the 9 h wall |

Together ≈ **0.24× of Train3's epoch**. The speed probe prints the real number
after 20 batches and **stops the run** if fewer than 20 epochs would fit, rather
than discovering it at hour four.

`RAM_BUDGET_GB = 6.0`: 4,053 × 64 × 128² = **4.25 GB**, so the whole corpus fits
and no study is dropped.

### Anti-overfit — the part that matters

Ordered by how much each is grounded in evidence, strongest first.

| knob | value | grounding |
|---|---|---|
| `MIXUP_ALPHA` / `MIXUP_P` | 0.4 / 0.5 | **manifold mixup on the pooled embeddings**, as the RSNA-2024 2nd place did. Mixup is the best-established defence against fitting corrupted labels, and the embedding is where it is cheap in an attention-MIL model |
| `LABEL_SMOOTH` | 0.05 | targets become 0.025 / 0.975. Report labels are ~82% concordant with gold, so a hard 1.0 asserts a certainty the label does not have |
| `EMA_DECAY` | 0.999 | a weight-space average of the trajectory. Both the raw and the EMA model are scored and saved — **which one is better is reported, not assumed** |
| `AUG_*` | see cell 1 | affine ±10° / ±12% zoom / ±6% shift, gamma 0.8–1.25, cutout. Plus `AUG_REVERSE_P = 0.5`, reversing slice order — taken from the 2nd-place solution |
| `BACKBONE_LR_MULT` | 0.25 | the backbone holds good ImageNet features; the noisy labels are allowed to nudge it, not rewrite it |
| `WEIGHT_DECAY` | 0.05 | AdamW, and **not** applied to norms or biases |
| `DROP_PATH` / `HEAD_DROP` | 0.1 / 0.3 | stochastic depth in the backbone, dropout in the attention head |
| `EARLY_STOP_PATIENCE` | 8 | on the **EMA** validation macro |
| `GRAD_CLIP` | 1.0 | one mislabelled batch cannot take a large step |
| `SELF_DISTIL_W` | **0.0 (off)** | the 2nd place "½ label + ½ pseudo-label" idea. Left off: unlike the rest it can entrench the model's own early errors, and it is unmeasured here. Turn it on only as a deliberate A/B |

**No horizontal flip.** On sagittal knee MRI a left-right flip swaps anterior
and posterior — it moves the ACL to where the PCL belongs. It is a free
augmentation on natural images and an anatomically false one here.

### Making overfitting visible

Every epoch prints **train macro, val macro, and the gap between them**, plus a
bootstrap 2.5% lower bound on the val macro. `TRAIN_PROBE_N = 300` studies keeps
the probe cheap. If the gap widens while val is flat, the regularisation is
losing and the run should stop — you can see that happen instead of inferring it
afterwards from a leaderboard score.

### Liveness

Progress prints are **time-based (every 30 s)**, not every N batches, so the
cadence is the same whether a batch takes 0.1 s or 3 s. Mid-run checkpoints
every 10 minutes. The previous session was lost to a 40-minute silence that was
indistinguishable from a hang.

---

## 4. What this arm is for

It is **not** trying to beat CoAtNet. The transformer arm loses to CoAtNet on
all twelve findings and still lifted the blend 0.928 → 0.935 — that came from
decorrelation. A third family is the one remaining lever with measured evidence
behind it. Session 4 fits its blend weight from the saved validation
predictions, so a weak-but-different arm is still worth having, and a
weak-and-similar one is not.

**The validation macro here is measured against report teachers, so it is not
comparable to 0.935 or to anything on the leaderboard.** Do not read it as a
score.


In [ ]:
# ---------------------------------------------------------------------------
# CONFIG -- every knob, with the reason it holds this value.
# ===========================================================================
# COST
# ===========================================================================
# Ordered by MEASURED speed on this hardware, not by ImageNet accuracy. On a
# T4 (sm_75) depthwise-separable convolutions fall into slow kernels:
# EfficientNet ran 21 min/epoch here on work ResNet50 did in 3.3. resnet34
# leads because it is the lightest plain-convolution net that still has real
# ImageNet weights, and its 512-d features shrink the attention head 4x.
ARCH_CANDIDATES = ["resnet34", "resnet50", "convnext_tiny",
                   "tf_efficientnetv2_s", "efficientnet_b0"]
ARCH_FORCE = None          # the preflight pairs arch to weights by counting
                           # matching tensors and prints the table; set this
                           # only to override that choice
STRICT_IMAGENET = True     # stop rather than silently spend a session training
                           # from scratch

TRAIN_PX = 128             # 4,053 x 64 x 128^2 = 4.25 GB in RAM
RAM_BUDGET_GB = 6.0        # ...so nothing is dropped
GROUP = 3                  # slices per window, matching the shipped pipeline
K_TRAIN = 4                # windows per study per step
K_VAL = 6
BATCH_STUDIES = 12
EPOCHS = 60                # the cosine horizon is cut to what actually fits
TIME_BUDGET_S = 4.0 * 3600
WORK = "/kaggle/working"
RESUME = True              # needs Settings -> Persistence -> Variables and Files

# One GPU on purpose. DataParallel replicates the model every step, which for a
# model this small costs more than it returns, and it makes mixup and gradient
# clipping semantics harder to reason about. Correctness first at this size.
USE_DP = False

# ===========================================================================
# ANTI-OVERFIT
# ===========================================================================
# The labels are report-derived and concordant with the gold adjudication only
# about 82% of the time, and the disagreement is SYSTEMATIC: the adjudicator
# grades borderline findings NEGATIVE, and a report does not encode that
# threshold. Fitting these labels closely is therefore fitting a known bias --
# which is exactly what V37 did, gaining on validation and losing 0.002 on the
# leaderboard. Everything below exists to prevent a repeat.

MIXUP_ALPHA = 0.4          # manifold mixup on the POOLED EMBEDDINGS, as the
MIXUP_P = 0.5              # RSNA-2024 2nd place did. Mixup is the best
                           # established defence against fitting corrupted
                           # labels; the embedding is where it is cheap here
LABEL_SMOOTH = 0.05        # targets 0.025 / 0.975 -- a hard 1.0 claims a
                           # certainty an 82%-concordant label does not have
EMA_DECAY = 0.999          # weight-space average of the trajectory; BOTH the
                           # raw and EMA models are scored, and which one wins
                           # is reported rather than assumed
DROP_PATH = 0.1            # stochastic depth in the backbone
HEAD_DROP = 0.3            # dropout in the attention head
BACKBONE_LR_MULT = 0.25    # let noisy labels nudge ImageNet features, not
                           # overwrite them
LR = 3e-4                  # head LR; the backbone gets a quarter of it
WEIGHT_DECAY = 0.05        # AdamW, and NOT applied to norms or biases
GRAD_CLIP = 1.0            # one mislabelled batch cannot take a large step
WARMUP_EPOCHS = 2
EARLY_STOP_PATIENCE = 8    # on the EMA validation macro

# Augmentation. No horizontal flip: on sagittal knee MRI a left-right flip
# swaps anterior and posterior, putting the ACL where the PCL belongs. Free on
# natural images, anatomically false here.
AUG_ROT = 10.0             # degrees
AUG_ZOOM = 0.12
AUG_SHIFT = 0.06
AUG_GAMMA = 0.22           # gamma in [1-g, 1+g], MRI intensity is uncalibrated
AUG_CUTOUT_P = 0.25
AUG_CUTOUT_FRAC = 0.25
AUG_REVERSE_P = 0.5        # reverse the slice order -- from the 2nd-place
                           # RSNA-2024 solution
AUG_DILATE_P = 0.25        # sample a window as t, t+2, t+4 instead of t, t+1, t+2

# The 2nd-place "0.5*loss_with_labels + 0.5*loss_with_pseudolabels" idea, using
# the EMA model as its own teacher. OFF by default: unlike everything above it
# can entrench the model's own early errors, and it is unmeasured on this
# problem. Turn it on only as a deliberate A/B against this run.
SELF_DISTIL_W = 0.0
SELF_DISTIL_START = 6      # epochs of plain training first, if it is enabled

# ===========================================================================
# DIAGNOSTICS -- so overfitting is visible while it happens
# ===========================================================================
TRAIN_PROBE_N = 300        # studies scored from the TRAIN split each epoch;
                           # train-minus-val is the overfitting readout
BOOT_N = 1000              # bootstrap resamples for the val-macro lower bound
VAL_FRACTION = 0.15
SPLIT_SEED = 2026

print("arm 3 (light) | %d windows/study | batch %d | <=%d epochs | %.1f h budget"
      % (K_TRAIN, BATCH_STUDIES, EPOCHS, TIME_BUDGET_S / 3600))
print("regularisers  | mixup %.2f p=%.2f | smooth %.3f | ema %.4f | dp %.2f "
      "| head-drop %.2f | wd %.3f"
      % (MIXUP_ALPHA, MIXUP_P, LABEL_SMOOTH, EMA_DECAY, DROP_PATH,
         HEAD_DROP, WEIGHT_DECAY))


In [ ]:
# ---------------------------------------------------------------------------
# PREFLIGHT -- the cache, the GPU, and ImageNet weights. Nothing reads DICOM.
import os, json, time, math, glob, gc, copy
import numpy as np

print("=" * 78)
print("PREFLIGHT")
print("=" * 78)
_missing, _warn = [], []

CACHE_DIR = None
for _d, _s, _f in os.walk("/kaggle/input"):
    _s[:] = [x for x in _s if x not in ("train_series", "test_series")]
    if "pixel_index.json" in _f and "pixels.npy" in _f:
        CACHE_DIR = _d
        break
if CACHE_DIR:
    _ix = json.load(open(os.path.join(CACHE_DIR, "pixel_index.json")))
    print("  pixel cache        : OK   %s" % CACHE_DIR)
    print("                       %d studies, %d slices, %d px"
          % (len(_ix["kept"]), _ix["slices"], _ix["px"]))
    for _n in ("masks.npy", "labels.npy"):
        if not os.path.exists(os.path.join(CACHE_DIR, _n)):
            _missing.append("%s is missing from the cache - session 1 did not "
                            "finish its last cell. Re-run it; it resumes." % _n)
else:
    print("  pixel cache        : MISSING")
    # Name WHICH notebook outputs are attached and what is in them. "Missing"
    # alone does not tell you that you picked the wrong one of two similarly
    # named notebooks - which is exactly how one session was lost.
    _nbroot = "/kaggle/input/notebooks"
    _seen = []
    if os.path.isdir(_nbroot):
        for _own in sorted(os.listdir(_nbroot)):
            _od = os.path.join(_nbroot, _own)
            if not os.path.isdir(_od):
                continue
            for _nbname in sorted(os.listdir(_od)):
                _p = os.path.join(_od, _nbname)
                if os.path.isdir(_p):
                    try:
                        _files = sorted(os.listdir(_p))[:5]
                    except OSError:
                        _files = []
                    _seen.append((_nbname, _files))
    if _seen:
        print("     notebook outputs attached, and what they hold:")
        for _nbname, _files in _seen:
            print("       %-28s %s" % (_nbname, ", ".join(_files) or "(empty)"))
        print("     -> none of these contains pixels.npy. Attach the session-1")
        print("        notebook instead; its output holds pixels.npy,")
        print("        masks.npy, labels.npy and pixel_index.json.")
    _missing.append(
        "The pixel cache from session 1. Add Input -> Notebook Output -> pick "
        "the notebook whose output listing above shows pixels.npy.")

import torch
_ng = torch.cuda.device_count()
print("  GPU                : %s" % ("OK   %d device(s)" % _ng if _ng else "NONE"))
if _ng == 0:
    _missing.append("A GPU. Settings -> Accelerator -> GPU T4 x2.")

import timm
print("  timm               : %s" % timm.__version__)

_WEIGHT_EXT = (".pth", ".pth.tar", ".bin", ".safetensors", ".pt")
_pool = []
for _d, _s, _f in os.walk("/kaggle/input"):
    _s[:] = [x for x in _s if x not in ("train_series", "test_series")]
    for _n in _f:
        if _n.lower().endswith(_WEIGHT_EXT):
            _pool.append(os.path.join(_d, _n))

_MINE = ("coatnet", "raptor", "arm3", "dinov")   # this project's own files


def _is_member(name):
    # the DINOv3 fold members are m_<hex>.pt, 89 MB, and must never be mistaken
    # for ImageNet weights
    import re as _re
    return bool(_re.match(r"^m_[0-9a-f]{2,}\.pt$", name))


_pool = [p for p in _pool
         if not any(k in os.path.basename(p).lower() for k in _MINE)
         and not _is_member(os.path.basename(p).lower())]
if _pool:
    print("  weight files found :")
    for _p in sorted(_pool, key=os.path.getsize, reverse=True)[:6]:
        print("      %-46s %5.0f MB"
              % (os.path.basename(_p)[:46], os.path.getsize(_p) / 1e6))


def _peek(path):
    """Return a flat state dict from whatever shape the file happens to be."""
    import torch as _t
    if path.endswith(".safetensors"):
        from safetensors.torch import load_file as _lf
        return _lf(path)
    sd = _t.load(path, map_location="cpu", weights_only=False)
    if hasattr(sd, "state_dict"):
        sd = sd.state_dict()
    for k in ("state_dict", "model", "model_state_dict", "net", "weights"):
        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict):
            sd = sd[k]
            break
    return sd if isinstance(sd, dict) else {}


def _match(sd, tgt):
    """Best tensor overlap over the prefixes checkpoints ship with."""
    best = 0
    for pre in ("", "module.", "model.", "backbone.", "net.", "encoder.",
                "features.", "trunk.", "base_model.", "resnet."):
        n = 0
        for k, v in sd.items():
            kk = k[len(pre):] if pre and k.startswith(pre) else k
            t = tgt.get(kk)
            if t is not None and hasattr(v, "shape") and tuple(v.shape) == tuple(t.shape):
                n += 1
        best = max(best, n)
    return best


# Pair every candidate architecture against every attached weight file and keep
# the pair that actually matches. Guessing an architecture from a file name has
# cost two sessions; counting tensors costs thirty seconds.
ARCH, WEIGHTS, _best, _fallback = None, None, 0.0, (None, None, 0.0)
if _pool:
    print("  matching architectures to weights:")
    _cands = [ARCH_FORCE] if ARCH_FORCE else ARCH_CANDIDATES
    # Sorted by size: an ImageNet backbone is the big file. Reading in os.walk
    # order once skipped the file that matched 100%% and reported "none found".
    _sds = {}
    for _p in sorted(_pool, key=os.path.getsize, reverse=True)[:10]:
        try:
            _sds[_p] = _peek(_p)
            if not _sds[_p]:
                del _sds[_p]
        except Exception as _e:
            print("      unreadable: %-30s %s"
                  % (os.path.basename(_p)[:30], type(_e).__name__))
    if not _sds:
        print("      no weight file could be opened as a state dict")
    for _a in _cands:
        try:
            _m = timm.create_model(_a, pretrained=False, num_classes=0, in_chans=3)
            _tgt = _m.state_dict()
            del _m
        except Exception as _e:
            print("      %-22s unavailable in timm (%s)" % (_a, type(_e).__name__))
            continue
        _bp, _bf, _bn = None, 0.0, 0
        for _p, _sd in _sds.items():
            _n = _match(_sd, _tgt)
            _f = _n / max(len(_tgt), 1)
            if _f > _bf:
                _bp, _bf, _bn = _p, _f, _n
        # One line per architecture, ALWAYS. A silent table is what hid the
        # last bug; zero matches is information too.
        print("      %-22s %-28s %3d/%3d  %3.0f%%"
              % (_a, os.path.basename(_bp)[:28] if _bp else "(nothing)",
                 _bn, len(_tgt), 100 * _bf))
        for _p, _sd in ((_bp, _sds[_bp]),) if _bp else ():
            _n, _f = _bn, _bf
            # Take the FIRST candidate that matches well, not the best match:
            # ARCH_CANDIDATES is ordered by measured speed on this GPU.
            if _f >= 0.80 and ARCH is None:
                ARCH, WEIGHTS, _best = _a, _p, _f
            elif ARCH is None and _f > _best:
                _fallback = (_a, _p, _f)
                _best = _f
        if ARCH is not None:
            break
    if ARCH is None and _best > 0.05:
        ARCH, WEIGHTS, _best = _fallback
        print("  no candidate reached 80%; falling back to the best partial match")
    if ARCH:
        print("  chosen             : %s  <- %s  (%.0f%% of tensors)"
              % (ARCH, os.path.basename(WEIGHTS), 100 * _best))

if ARCH is None:
    ARCH = ARCH_FORCE or ARCH_CANDIDATES[0]
    print("  ImageNet weights   : NONE FOUND -> training %s from scratch" % ARCH)
    _warn.append(
        "No ImageNet weights were found, so the backbone starts from random. "
        "That is materially weaker. Add Input -> Models, search 'timm', attach "
        "one of: " + ", ".join(ARCH_CANDIDATES) + ".")
else:
    print("  ImageNet weights   : OK   %s" % os.path.basename(WEIGHTS))
    print("  architecture       : %s" % ARCH)

print("=" * 78)
if _missing:
    print("ATTACH THESE, THEN RE-RUN")
    for _i, _m in enumerate(_missing, 1):
        print("  %d. %s" % (_i, _m))
    print("=" * 78)
    raise SystemExit("preflight failed: %d input(s) missing" % len(_missing))
for _w in _warn:
    print("WARNING: %s" % _w)
print("preflight OK -- architecture %s" % ARCH)
print("=" * 78)


In [ ]:
# ---------------------------------------------------------------------------
# The cache, into RAM once, and the valid window range per study.
IX = json.load(open(os.path.join(CACHE_DIR, "pixel_index.json")))
LAB = IX["labels"]
PX, MAXS = int(IX["px"]), int(IX["slices"])
UIDS = list(IX["kept"])
N = len(UIDS)

_MM = np.load(os.path.join(CACHE_DIR, "pixels.npy"), mmap_mode="r")
MSK = np.asarray(np.load(os.path.join(CACHE_DIR, "masks.npy"), mmap_mode="r"))
Y = np.load(os.path.join(CACHE_DIR, "labels.npy")).astype(np.float32)
assert _MM.shape[0] >= N and Y.shape[0] == N, "cache arrays disagree on length"
print("cache on disk: %s uint8, labels %s" % (_MM.shape, Y.shape))

_per = MAXS * TRAIN_PX * TRAIN_PX
_fit = int(RAM_BUDGET_GB * 1e9 // _per)
if _fit < N:
    print("RAM budget %.1f GB holds %d of %d studies at %d px"
          % (RAM_BUDGET_GB, _fit, N, TRAIN_PX))
    N = _fit
    UIDS, Y, MSK = UIDS[:N], Y[:N], MSK[:N]
print("loading %d studies into RAM at %d px = %.2f GB (one sequential pass)"
      % (N, TRAIN_PX, N * _per / 1e9))

import cv2

_PIXFILE = os.path.join(CACHE_DIR, "pixels.npy")
_FD = os.open(_PIXFILE, os.O_RDONLY)


def drop_page_cache():
    """Tell the kernel to forget this file's cached pages.

    Reading the cache sequentially fills the page cache, and a container
    charges that cache against its memory limit. `del` and gc release the
    MAPPING, not the cache - which is why two earlier fixes changed nothing.
    """
    try:
        os.posix_fadvise(_FD, 0, 0, os.POSIX_FADV_DONTNEED)
    except Exception:
        pass


def mem_gb():
    try:
        with open("/proc/meminfo") as f:
            m = {l.split(":")[0]: int(l.split()[1]) for l in f}
        return (m["MemTotal"] - m["MemAvailable"]) / 1e6, m["MemTotal"] / 1e6
    except Exception:
        return float("nan"), float("nan")


PIX = np.empty((N, MAXS, TRAIN_PX, TRAIN_PX), np.uint8)
_t0 = time.time()
for _i in range(N):
    _v = np.asarray(_MM[_i])                       # sequential, one study at a time
    if _v.shape[-1] == TRAIN_PX:
        PIX[_i] = _v
    else:
        for _z in range(MAXS):
            PIX[_i, _z] = cv2.resize(_v[_z], (TRAIN_PX, TRAIN_PX),
                                     interpolation=cv2.INTER_AREA)
    if (_i + 1) % 250 == 0:
        drop_page_cache()          # keep the cache from ever growing large
        _el = time.time() - _t0
        _u, _tt = mem_gb()
        print("  %4d/%d  %.1f min  eta %.1f min  |  RAM %.1f/%.1f GB"
              % (_i + 1, N, _el / 60, (N - _i - 1) * _el / (_i + 1) / 60, _u, _tt),
              flush=True)
del _MM
gc.collect()
drop_page_cache()
_u, _tt = mem_gb()
print("in RAM: %s uint8, %.2f GB, loaded in %.1f min  |  RAM now %.1f/%.1f GB"
      % (PIX.shape, PIX.nbytes / 1e9, (time.time() - _t0) / 60, _u, _tt))


def valid_range(mask):
    """Mirrors _eval_centers: windows live between the first and last filled slot."""
    v = np.flatnonzero(np.asarray(mask) > 0)
    if len(v) < GROUP:
        return 0, max(0, MAXS - GROUP)
    lo, hi = int(v.min()), int(v.max())
    return lo, max(lo, hi - GROUP + 1)


RANGES = np.array([valid_range(MSK[i]) for i in range(N)], np.int32)
_span = RANGES[:, 1] - RANGES[:, 0] + 1
print("windows available per study: median %d, min %d, max %d"
      % (int(np.median(_span)), int(_span.min()), int(_span.max())))

rng = np.random.default_rng(SPLIT_SEED)
order = rng.permutation(N)
nv = int(round(VAL_FRACTION * N))
VA, TR = np.sort(order[:nv]), np.sort(order[nv:])
# A FIXED subsample of train, scored every epoch. train-minus-val is the
# overfitting readout, and it must be the same studies every epoch to be
# comparable across them.
TP = np.sort(rng.choice(TR, size=min(TRAIN_PROBE_N, len(TR)), replace=False))
print("split: %d train, %d val (seed %d, fixed across resumes)"
      % (len(TR), len(VA), SPLIT_SEED))
print("train probe: %d studies, fixed, scored every epoch" % len(TP))
_pos = np.nanmean(Y >= 0.5, axis=0)
print("label prevalence: %.2f-%.2f across the twelve findings"
      % (_pos.min(), _pos.max()))


In [ ]:
# ---------------------------------------------------------------------------
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

torch.backends.cudnn.benchmark = True
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def build_backbone(arch):
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "eva", "beit"))
    kw = dict(pretrained=False, num_classes=0, in_chans=3)
    kw.update(dict(global_pool="token", dynamic_img_size=True) if is_vit
              else dict(global_pool="avg"))
    try:                       # stochastic depth, if this family supports it
        return timm.create_model(arch, drop_path_rate=DROP_PATH, **kw)
    except TypeError:
        print("  (%s takes no drop_path_rate; stochastic depth is off)" % arch)
        return timm.create_model(arch, **kw)


class Arm3(nn.Module):
    """Backbone over windows, then the pipeline's attention pooling.

    embed() and head() are separate so manifold mixup can happen BETWEEN them,
    on the pooled embeddings, which is where the RSNA-2024 2nd-place solution
    mixed and where it is cheap in an attention-MIL model.
    """

    def __init__(self, arch, n=12, drop=0.3):
        super().__init__()
        self.backbone = build_backbone(arch)
        d = self.backbone.num_features
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
        self.att = nn.Sequential(nn.Linear(d, 256), nn.Tanh(),
                                 nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, d))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)

    def embed(self, x):                        # (B, K, 3, H, W) -> (B, K, d)
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).view(B, K, -1)
        return self.norm(f)

    def head(self, h):                         # (B, K, d) -> (B, 12)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, self.drop(h))
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.embed(x))


model = Arm3(ARCH, drop=HEAD_DROP)


def load_imagenet(path, backbone):
    """Match a checkpoint against the backbone, trying the prefixes people
    actually ship. Reports what it found rather than silently loading 0%."""
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if hasattr(sd, "state_dict"):                 # a pickled nn.Module
        sd = sd.state_dict()
    for k in ("state_dict", "model", "model_state_dict", "net", "weights"):
        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict):
            sd = sd[k]
            break
    if not isinstance(sd, dict):
        raise TypeError("checkpoint is a %s, not a state dict" % type(sd).__name__)
    tgt = backbone.state_dict()
    best, best_p = {}, ""
    for p in ("", "module.", "model.", "backbone.", "net.", "encoder.",
              "features.", "trunk.", "base_model.", "resnet."):
        cand = {}
        for k, v in sd.items():
            kk = k[len(p):] if p and k.startswith(p) else k
            t = tgt.get(kk)
            if t is not None and hasattr(v, "shape") and tuple(v.shape) == tuple(t.shape):
                cand[kk] = v
        if len(cand) > len(best):
            best, best_p = cand, p
    return sd, tgt, best, best_p


loaded = 0
if WEIGHTS:
    try:
        sd, tgt, take, pref = load_imagenet(WEIGHTS, model.backbone)
        model.backbone.load_state_dict(take, strict=False)
        loaded = len(take)
        print("ImageNet init: %d of %d backbone tensors matched (%.0f%%)%s"
              % (loaded, len(tgt), 100.0 * loaded / max(len(tgt), 1),
                 "" if not pref else "  [stripped prefix %r]" % pref))
        if loaded < 0.5 * len(tgt):
            print("WARNING: fewer than half matched - the file does not look "
                  "like %s. Training would be mostly from scratch." % ARCH)
            print("  checkpoint keys : %s" % ", ".join(list(sd.keys())[:6]))
            print("  %s expects      : %s" % (ARCH, ", ".join(list(tgt.keys())[:6])))
            if STRICT_IMAGENET:
                raise SystemExit(
                    "Stopping rather than burning a session on a scratch run. "
                    "Attach weights that match %s, or set STRICT_IMAGENET = "
                    "False to train from scratch deliberately." % ARCH)
    except SystemExit:
        raise
    except Exception as exc:
        print("could not load %s (%s: %s)"
              % (os.path.basename(WEIGHTS), type(exc).__name__, exc))
        if STRICT_IMAGENET:
            raise SystemExit("Stopping: no usable ImageNet weights. Set "
                             "STRICT_IMAGENET = False to train from scratch.")
else:
    print("no ImageNet weights found")
    if STRICT_IMAGENET:
        raise SystemExit("Stopping: no ImageNet weights attached. Set "
                         "STRICT_IMAGENET = False to train from scratch.")

print("backbone %s | feature dim %d | %.1fM params"
      % (ARCH, model.backbone.num_features,
         sum(p.numel() for p in model.parameters()) / 1e6))
model = model.to(DEV)
core = model
if USE_DP and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("DataParallel over %d GPUs" % torch.cuda.device_count())
    print("  NOTE: mixup steps call core.embed/core.head directly, so those "
          "steps run on ONE GPU. Correct, but ~%d%% of steps lose the second."
          % int(100 * MIXUP_P))
else:
    print("single GPU on purpose (USE_DP=%s): replication costs more than it "
          "returns at this model size, and mixup/clipping stay exact"
          % USE_DP)


class EMA:
    """Weight-space average of the training trajectory.

    The decay is ramped in, so the average is usable from the first epochs
    instead of lagging a long way behind for the first few thousand steps.
    """

    def __init__(self, m, decay):
        self.decay, self.n = decay, 0
        self.shadow = copy.deepcopy(m).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, m):
        self.n += 1
        d = min(self.decay, (1.0 + self.n) / (10.0 + self.n))
        for s, q in zip(self.shadow.state_dict().values(), m.state_dict().values()):
            if s.dtype.is_floating_point:
                s.mul_(d).add_(q.detach(), alpha=1.0 - d)
            else:
                s.copy_(q)


ema = EMA(core, EMA_DECAY)
print("EMA shadow initialised, decay %.4f (ramped)" % EMA_DECAY)


class Bags(Dataset):
    def __init__(self, idx, k, train):
        self.idx, self.k, self.train = np.asarray(idx), int(k), bool(train)

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        s = int(self.idx[i])
        lo, hi = int(RANGES[s, 0]), int(RANGES[s, 1])
        # Window stride. A dilated window covers more of the joint for the same
        # three slices, and it is a second axis of augmentation that costs
        # nothing. But hi is the last start for stride 1: it already equals
        # last_filled - (GROUP - 1). A stride-2 window spans twice as far, so
        # its last legal start is hi - (GROUP - 1), and the bound has to come
        # from hi, NOT from lo. Deriving it from lo lets a study whose series
        # ends early -- exactly the studies the slot mask exists for -- sample
        # windows made largely of zero padding, silently, for the whole run.
        d = 1
        if self.train and np.random.rand() < AUG_DILATE_P:
            hi2 = hi - (GROUP - 1)
            if hi2 >= lo:
                d, hi = 2, hi2
        span = max(hi - lo + 1, 1)
        if self.train:
            starts = np.random.randint(lo, lo + span, size=self.k)
        else:
            starts = (lo + np.linspace(0, span - 1, self.k)).round().astype(int)
        # belt and braces: even a degenerate RANGES row cannot address past the
        # end of the volume
        starts = np.clip(starts, 0, MAXS - 1 - (GROUP - 1) * d)
        w = np.stack([PIX[s, t:t + GROUP * d:d]
                      for t in starts]).astype(np.float32) / 255.0
        if self.train and np.random.rand() < AUG_REVERSE_P:
            # Reverse the slice order -- taken from the RSNA-2024 2nd place.
            # It is a real symmetry of a stack read in either direction, unlike
            # a left-right flip, which would move the ACL to the PCL's place.
            w = w[:, ::-1].copy()
        return torch.from_numpy(w), torch.from_numpy(Y[s]), s


def collate(b):
    return (torch.stack([x[0] for x in b]), torch.stack([x[1] for x in b]),
            torch.tensor([x[2] for x in b]))


def norm_batch(w):
    B, K = w.shape[:2]
    return ((w.flatten(0, 1) - _MEAN.to(w.device)) / _STD.to(w.device)).view(
        B, K, GROUP, w.shape[-2], w.shape[-1])


def augment(w):
    """Geometry and intensity, on the GPU, in fp32, before normalisation.

    Deliberately NO horizontal flip: on sagittal knee MRI it swaps anterior and
    posterior. Every transform here is one the same knee could genuinely have
    produced -- a different table position, a different receiver gain.
    """
    B, K = w.shape[:2]
    x = w.flatten(0, 1)                              # (B*K, 3, H, W)
    n, _, H, W = x.shape
    dev = x.device

    s = torch.empty(n, device=dev).uniform_(1.0 - AUG_ZOOM, 1.0 + AUG_ZOOM)
    a = torch.empty(n, device=dev).uniform_(-AUG_ROT, AUG_ROT) * (math.pi / 180.0)
    tx = torch.empty(n, device=dev).uniform_(-AUG_SHIFT, AUG_SHIFT)
    ty = torch.empty(n, device=dev).uniform_(-AUG_SHIFT, AUG_SHIFT)
    cos, sin = torch.cos(a) / s, torch.sin(a) / s
    th = torch.zeros(n, 2, 3, device=dev)
    th[:, 0, 0], th[:, 0, 1], th[:, 0, 2] = cos, -sin, tx
    th[:, 1, 0], th[:, 1, 1], th[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(th, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, align_corners=False, padding_mode="border")

    # MRI intensity is not calibrated across scanners, so gamma and gain are
    # nuisance parameters the model should be invariant to, not signal.
    g = torch.empty(n, 1, 1, 1, device=dev).uniform_(1.0 - AUG_GAMMA, 1.0 + AUG_GAMMA)
    x = x.clamp(0.0, 1.0).pow(g)

    if AUG_CUTOUT_P > 0:
        ch, cw = int(H * AUG_CUTOUT_FRAC), int(W * AUG_CUTOUT_FRAC)
        y0 = torch.randint(0, max(H - ch, 1), (n,), device=dev)
        x0 = torch.randint(0, max(W - cw, 1), (n,), device=dev)
        ys = torch.arange(H, device=dev).view(1, H, 1)
        xs = torch.arange(W, device=dev).view(1, 1, W)
        hit = (torch.rand(n, device=dev) < AUG_CUTOUT_P).view(n, 1, 1)
        box = ((ys >= y0.view(n, 1, 1)) & (ys < (y0 + ch).view(n, 1, 1))
               & (xs >= x0.view(n, 1, 1)) & (xs < (x0 + cw).view(n, 1, 1)) & hit)
        x = x.masked_fill(box.unsqueeze(1), 0.0)
    return x.view(B, K, GROUP, H, W)


# num_workers=0: PIX is already in RAM, so a worker only adds a fork and one
# more thing to account for. Three sessions have died to memory accounting.
NW = 0
dl_tr = DataLoader(Bags(TR, K_TRAIN, True), batch_size=BATCH_STUDIES, shuffle=True,
                   num_workers=NW, collate_fn=collate, drop_last=True,
                   pin_memory=False)
dl_va = DataLoader(Bags(VA, K_VAL, False), batch_size=max(2, BATCH_STUDIES // 2),
                   shuffle=False, num_workers=NW, collate_fn=collate,
                   pin_memory=False)
dl_tp = DataLoader(Bags(TP, K_VAL, False), batch_size=max(2, BATCH_STUDIES // 2),
                   shuffle=False, num_workers=NW, collate_fn=collate,
                   pin_memory=False)
print("loaders: %d train batches, %d val batches, %d train-probe batches"
      % (len(dl_tr), len(dl_va), len(dl_tp)))


In [ ]:
# ---------------------------------------------------------------------------
# Train. Checkpoints are time-based, progress prints are time-based, and the
# train-minus-val gap is printed every epoch so overfitting is visible while it
# is happening rather than inferred afterwards from a leaderboard score.
CKPT = os.path.join(WORK, "arm3.pt")


def param_groups(m, lr, wd, bb_mult):
    """Backbone at a fraction of the head's LR; no decay on norms or biases.

    The backbone carries ImageNet features that are better than anything these
    labels can teach it. It should be nudged, not rewritten.
    """
    g = {"bb_d": [], "bb_n": [], "hd_d": [], "hd_n": []}
    for n_, p in m.named_parameters():
        if not p.requires_grad:
            continue
        k = ("bb_" if n_.startswith("backbone.") else "hd_") + \
            ("n" if (p.ndim <= 1 or n_.endswith(".bias")) else "d")
        g[k].append(p)
    return [{"params": g["bb_d"], "lr": lr * bb_mult, "weight_decay": wd},
            {"params": g["bb_n"], "lr": lr * bb_mult, "weight_decay": 0.0},
            {"params": g["hd_d"], "lr": lr, "weight_decay": wd},
            {"params": g["hd_n"], "lr": lr, "weight_decay": 0.0}]


opt = torch.optim.AdamW(param_groups(core, LR, WEIGHT_DECAY, BACKBONE_LR_MULT))
BASE_LR = [pg["lr"] for pg in opt.param_groups]
scaler = torch.amp.GradScaler("cuda", enabled=str(DEV).startswith("cuda"))
lossf = nn.BCEWithLogitsLoss(reduction="none")
start_epoch, best, hist, gstep = 0, -1.0, [], 0


def lr_scale(step, total, warmup):
    """Linear warmup, then cosine down to 2% of base."""
    if step < warmup:
        return (step + 1) / max(warmup, 1)
    t = (step - warmup) / max(total - warmup, 1)
    return 0.02 + 0.98 * 0.5 * (1.0 + math.cos(math.pi * min(t, 1.0)))


def soft_target(y):
    """Label smoothing on the observed entries only.

    These labels agree with the gold adjudication about 82% of the time, so a
    hard 1.0 asserts a confidence the label does not carry. NaN entries are
    unobserved and are masked out of the loss entirely rather than guessed.
    """
    m = torch.isfinite(y).float()
    t = torch.nan_to_num(y, nan=0.5).clamp(0.0, 1.0)
    return t * (1.0 - LABEL_SMOOTH) + 0.5 * LABEL_SMOOTH, m


def masked_bce(out, t, m):
    return (lossf(out, t) * m).sum() / m.sum().clamp(min=1)


# --- resume ----------------------------------------------------------------
_resume_from = CKPT if os.path.exists(CKPT) else None
if RESUME and _resume_from is None:
    for _d, _s, _f in os.walk("/kaggle/input"):
        _s[:] = [x for x in _s if x not in ("train_series", "test_series")]
        if "arm3.pt" in _f:
            _resume_from = os.path.join(_d, "arm3.pt")
            print("found a checkpoint in the attached inputs: %s" % _resume_from)
            break
if RESUME and _resume_from:
    try:
        ck = torch.load(_resume_from, map_location="cpu", weights_only=False)
        if ck.get("arch") == ARCH:
            core.load_state_dict(ck["model"])
            if ck.get("model_ema"):
                ema.shadow.load_state_dict(ck["model_ema"])
                ema.n = ck.get("ema_n", 0)
            opt.load_state_dict(ck["opt"])
            start_epoch = ck["epoch"] + 1
            best, hist = ck["best"], ck.get("hist", [])
            gstep = ck.get("gstep", 0)
            print("resuming at epoch %d, best EMA val macro %.4f"
                  % (start_epoch, best))
        else:
            print("checkpoint is for %s, not %s; starting fresh"
                  % (ck.get("arch"), ARCH))
    except Exception as exc:
        print("checkpoint unreadable (%s); starting fresh" % type(exc).__name__)


def predict(net, dl):
    net.eval()
    P, S = [], []
    with torch.no_grad():
        for w, y, s in dl:
            w = norm_batch(w.to(DEV, non_blocking=True))
            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=str(DEV).startswith("cuda")):
                P.append(torch.sigmoid(net(w).float()).cpu().numpy())
            S.append(s.numpy())
    return np.concatenate(P), np.concatenate(S)


def macro_auc(P, S):
    yv = Y[S]
    per = []
    for j in range(12):
        m = np.isfinite(yv[:, j])
        per.append(roc_auc_score((yv[m, j] >= 0.5).astype(int), P[m, j])
                   if m.sum() > 20 and len(np.unique(yv[m, j] >= 0.5)) > 1
                   else np.nan)
    return float(np.nanmean(per)), per


def boot_lo(P, S, n=None, seed=7):
    """Study-level bootstrap 2.5% lower bound on the macro AUC.

    Without it a +0.002 wobble between epochs reads as progress. With ~600 val
    studies the band is wide, and it should be.
    """
    n = BOOT_N if n is None else n
    r = np.random.default_rng(seed)
    yv, out = Y[S], []
    for _ in range(n):
        b = r.integers(0, len(S), len(S))
        per = []
        for j in range(12):
            yy, pp = yv[b, j], P[b, j]
            m = np.isfinite(yy)
            if m.sum() > 20 and len(np.unique(yy[m] >= 0.5)) > 1:
                per.append(roc_auc_score((yy[m] >= 0.5).astype(int), pp[m]))
        if per:
            out.append(float(np.mean(per)))
    return float(np.percentile(out, 2.5)) if out else float("nan")


# --- speed probe -----------------------------------------------------------
# Twenty batches, then project, and STOP if the configuration cannot finish.
# Three sessions were spent discovering the epoch time after committing to it.
model.train()
# The probe takes 20 REAL optimiser steps. Run them at 1% of base LR so they
# do not land as 20 full-rate steps on a freshly initialised head before the
# warmup has begun; the LR is restored by the scheduler on the first real step.
for _i, _pg in enumerate(opt.param_groups):
    _pg["lr"] = BASE_LR[_i] * 0.01
_p0, _pn = time.time(), 0
for w, y, _ in dl_tr:
    w = norm_batch(augment(w.to(DEV, non_blocking=True)))
    t, m = soft_target(y.to(DEV, non_blocking=True))
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16,
                        enabled=str(DEV).startswith("cuda")):
        l = masked_bce(model(w).float(), t, m)
    scaler.scale(l).backward()
    scaler.step(opt)
    scaler.update()
    _pn += 1
    if _pn >= 20:
        break
_pb = (time.time() - _p0) / max(_pn, 1)
_pe = _pb * len(dl_tr) / 60.0
# Validation is not a rounding error: at K_VAL windows it is its own pass over
# the data, and both the val split and the train probe are scored, for the raw
# model and the EMA model. The old 15% guess is why "27 min/epoch" was really 42.
_pv = _pb * (len(dl_va) + len(dl_tp)) * 2 * (K_VAL / max(K_TRAIN, 1)) / 3.0 / 60.0
_pf = TIME_BUDGET_S / 60.0 / max(_pe + _pv, 1e-9)
print("")
print("speed probe: %.2f s/batch -> train %.1f min + eval %.1f min = %.1f min/epoch"
      % (_pb, _pe, _pv, _pe + _pv))
print("             -> about %d epochs in %.1f h" % (int(_pf), TIME_BUDGET_S / 3600))
if _pf < 20:
    print("  TOO SLOW for a light run. Options, cheapest first:")
    print("    - ARCH_FORCE = \"resnet34\"; depthwise-heavy nets (EfficientNet)")
    print("      hit slow kernels on this GPU and ran 6x slower for no gain")
    print("    - TRAIN_PX = 112, or K_TRAIN = 3, or TRAIN_PROBE_N = 150")
    print("  Stopping now rather than spending the session to learn this.")
    raise SystemExit("speed probe: %.1f min/epoch is too slow" % (_pe + _pv))
# Anneal over what actually FITS. A cosine that never completes leaves the LR
# high and the model under-annealed at the point the budget runs out.
SCHED_EPOCHS = max(8, min(EPOCHS, int(_pf)))
TOTAL_STEPS = SCHED_EPOCHS * len(dl_tr)
WARMUP_STEPS = WARMUP_EPOCHS * len(dl_tr)
print("  fast enough. Annealing the cosine over %d epochs (%d steps), "
      "warmup %d." % (SCHED_EPOCHS, TOTAL_STEPS, WARMUP_STEPS))
print("")
opt.zero_grad(set_to_none=True)

t0, bad = time.time(), 0
_last_ck = time.time()
for ep in range(start_epoch, EPOCHS):
    if time.time() - t0 > TIME_BUDGET_S:
        print("time budget reached before epoch %d; checkpoint is saved" % ep)
        break
    model.train()
    tot, nb, nmix = 0.0, 0, 0
    _e0 = _last_print = time.time()
    for w, y, _ in dl_tr:
        for i_, pg in enumerate(opt.param_groups):
            pg["lr"] = BASE_LR[i_] * lr_scale(gstep, TOTAL_STEPS, WARMUP_STEPS)
        w = norm_batch(augment(w.to(DEV, non_blocking=True)))
        t, m = soft_target(y.to(DEV, non_blocking=True))
        opt.zero_grad(set_to_none=True)
        use_mix = MIXUP_ALPHA > 0 and np.random.rand() < MIXUP_P
        with torch.autocast("cuda", dtype=torch.float16,
                            enabled=str(DEV).startswith("cuda")):
            if use_mix:
                # Manifold mixup: mix the POOLED EMBEDDINGS, not the pixels.
                # This is where the RSNA-2024 2nd place mixed, and in an
                # attention-MIL model it is both cheaper and more meaningful
                # than mixing images. The loss keeps the two label sets
                # separate so each one's NaN mask stays correct.
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm = torch.randperm(w.shape[0], device=w.device)
                h = core.embed(w)
                out = core.head(lam * h + (1.0 - lam) * h[perm]).float()
                loss = (lam * masked_bce(out, t, m)
                        + (1.0 - lam) * masked_bce(out, t[perm], m[perm]))
                nmix += 1
            else:
                out = model(w).float()
                if SELF_DISTIL_W > 0 and ep >= SELF_DISTIL_START:
                    # The 2nd-place "half label, half pseudo-label" idea, with
                    # the EMA model as its own teacher. Only on OBSERVED
                    # entries: inventing supervision where the label is missing
                    # is where confirmation bias lives.
                    with torch.no_grad():
                        p_ema = torch.sigmoid(ema.shadow(w).float())
                    t = (1.0 - SELF_DISTIL_W) * t + SELF_DISTIL_W * p_ema
                loss = masked_bce(out, t, m)
        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(core.parameters(), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        ema.update(core)
        tot += float(loss.detach())
        nb += 1
        gstep += 1
        # TIME-based, not every N batches: the cadence is then the same whether
        # a batch takes 0.1 s or 3 s. A 40-minute silence in a batch-run
        # notebook is indistinguishable from a hang, which is how one session
        # was lost -- not to a crash, but to having no way to tell.
        if time.time() - _last_print > 30:
            _el = time.time() - _e0
            print("    epoch %2d  batch %4d/%d  loss %.4f  lr %.2e  %.0f s  "
                  "eta %.1f min" % (ep, nb, len(dl_tr), tot / nb,
                                    opt.param_groups[-1]["lr"], _el,
                                    (len(dl_tr) - nb) * _el / nb / 60), flush=True)
            _last_print = time.time()
        if time.time() - _last_ck > 600:
            torch.save({"arch": ARCH, "epoch": ep - 1, "best": best, "hist": hist,
                        "model": core.state_dict(),
                        "model_ema": ema.shadow.state_dict(), "ema_n": ema.n,
                        "opt": opt.state_dict(), "gstep": gstep,
                        "px": TRAIN_PX, "group": GROUP, "k_val": K_VAL,
                        "labels": LAB, "uids": UIDS, "val_idx": VA.tolist(),
                        "split_seed": SPLIT_SEED, "partial": True}, CKPT)
            _last_ck = time.time()
        if time.time() - t0 > TIME_BUDGET_S:
            break

    # --- score the raw model and the EMA model, and probe the train split ---
    Pr, Sr = predict(model, dl_va)
    mac_raw, _ = macro_auc(Pr, Sr)
    Pe, Se = predict(ema.shadow, dl_va)
    mac_ema, per = macro_auc(Pe, Se)
    Pt, St = predict(ema.shadow, dl_tp)
    mac_tr, _ = macro_auc(Pt, St)
    gc.collect()

    macro = mac_ema
    hist.append({"epoch": ep, "loss": tot / max(nb, 1), "val_raw": mac_raw,
                 "val_ema": mac_ema, "train_probe": mac_tr,
                 "gap": mac_tr - mac_ema})
    star = ""
    if macro > best:
        best, star, bad = macro, "  <- best", 0
        np.save(os.path.join(WORK, "arm3_val_pred.npy"), Pe)
        np.save(os.path.join(WORK, "arm3_val_idx.npy"), Se)
        np.save(os.path.join(WORK, "arm3_val_pred_raw.npy"), Pr)
    else:
        bad += 1
    torch.save({"arch": ARCH, "epoch": ep, "best": best, "hist": hist,
                "model": core.state_dict(),
                "model_ema": ema.shadow.state_dict(), "ema_n": ema.n,
                "opt": opt.state_dict(), "gstep": gstep,
                "px": TRAIN_PX, "group": GROUP, "k_val": K_VAL,
                "labels": LAB, "uids": UIDS, "val_idx": VA.tolist(),
                "split_seed": SPLIT_SEED}, CKPT)
    _last_ck = time.time()
    _u, _tt = mem_gb()
    _g = (torch.cuda.max_memory_allocated() / 1e9
          if str(DEV).startswith("cuda") else 0.0)
    print("epoch %2d  loss %.4f  val raw %.4f  val EMA %.4f%s  | train %.4f  "
          "gap %+.4f  | mix %d%%  %.1f min  RAM %.1f/%.1f  GPU %.1f GB"
          % (ep, tot / max(nb, 1), mac_raw, mac_ema, star, mac_tr,
             mac_tr - mac_ema, int(100 * nmix / max(nb, 1)),
             (time.time() - t0) / 60, _u, _tt, _g), flush=True)
    if bad >= EARLY_STOP_PATIENCE:
        print("early stop: %d epochs without a new best EMA macro" % bad)
        break


In [ ]:
# ---------------------------------------------------------------------------
Pe, Se = predict(ema.shadow, dl_va)
macro, per = macro_auc(Pe, Se)
Pr, Sr = predict(model, dl_va)
mac_raw, _ = macro_auc(Pr, Sr)
lo = boot_lo(Pe, Se)

print("=" * 78)
print("ARM 3 (light) -- %s" % ARCH)
print("=" * 78)
print("  epochs trained    : %d" % len(hist))
print("  best EMA val macro: %.4f" % best)
print("  final EMA         : %.4f   bootstrap 2.5%% lower bound %.4f"
      % (macro, lo))
print("  final raw         : %.4f   %s"
      % (mac_raw, "EMA is better" if macro > mac_raw else
         "raw is better -- prefer arm3_val_pred_raw.npy in session 4"))
print("")
print("  per-finding (EMA, final epoch):")
for j, t in enumerate(LAB):
    print("    %-19s %.4f" % (t, per[j]))
print("")
if hist:
    print("  epoch    loss   val_ema   train   gap")
    for h in hist[-8:]:
        print("    %3d  %.4f   %.4f  %.4f  %+.4f"
              % (h["epoch"], h["loss"], h["val_ema"], h["train_probe"], h["gap"]))
    print("")

_g = [h["gap"] for h in hist]
_v = [h["val_ema"] for h in hist]
print("  READING THE GAP")
if len(_g) >= 4:
    _early, _late = float(np.mean(_g[:3])), float(np.mean(_g[-3:]))
    print("    train-minus-val: %+.4f early -> %+.4f late" % (_early, _late))
    if _late > _early + 0.02 and _v[-1] <= max(_v[:-1]):
        print("    -> the gap is widening while val is not improving. That is")
        print("       overfitting the report labels, which is exactly what cost")
        print("       V37 its submission. Raise MIXUP_P, LABEL_SMOOTH or")
        print("       HEAD_DROP before spending another session.")
    elif _late > _early + 0.02:
        print("    -> the gap is widening but val is still rising. Normal;")
        print("       watch it, and stop when val flattens.")
    else:
        print("    -> the gap is stable. The regularisation is holding.")
if len(_v) >= 4:
    _last = _v[-4:]
    print("    last four val_ema: " + ", ".join("%.4f" % x for x in _last))
    print("    -> %s" % ("still climbing; re-running this notebook resumes and "
                         "adds epochs" if _last[-1] > max(_last[:-1]) else
                         "plateaued; go to session 4"))
print("")
print("#" * 78)
print("# CHECKPOINT : %s   (holds BOTH 'model' and 'model_ema')" % CKPT)
print("# VAL PRED   : arm3_val_pred.npy (EMA) + arm3_val_pred_raw.npy + "
      "arm3_val_idx.npy")
print("#" * 78)
print("""
WHAT TO DO NEXT

  Still climbing -> re-run this same notebook. It resumes from the checkpoint,
  restores the EMA shadow and the LR step count, and continues.

  Plateaued -> session 4: attach arm3.pt, run it as a third arm, and fit its
  blend weight from the saved validation predictions rather than guessing one.

HOW TO READ THE NUMBERS, AND HOW NOT TO

  The validation macro is measured against the REPORT teachers, not gold. It is
  NOT comparable to 0.935, to the leaderboard, or to the CoAtNet arm's 0.9214.
  Do not read it as a score.

  What decides whether this arm is worth blending is not its strength but its
  DECORRELATION from the other two. The transformer arm loses to CoAtNet on all
  twelve findings and still added +0.007. Session 4 measures that directly, by
  fitting the weight on the validation predictions saved above.

  A higher val macro here is therefore NOT automatically good news. If this arm
  reaches a high val macro by fitting the report labels closely, it has learned
  the same systematic bias the teachers carry -- borderline findings graded
  positive that the adjudicator grades negative -- and it will not transfer.
  The gap line above is the honest readout, not the macro.

WHAT HAS AND HAS NOT BEEN MEASURED

  Measured on this project: the +0.007 from a decorrelated arm; the -0.002 from
  a head retrained on report labels; 384 px carrying +0.00302 that a frozen head
  cannot use.

  NOT measured here: every regulariser in this notebook. Mixup, EMA, label
  smoothing and the augmentation set are standard practice and two of them come
  from the RSNA-2024 2nd-place solution, but none has been A/B'd on THIS
  problem. This run is the baseline that makes such an A/B possible; it is not
  evidence by itself, and no score should be predicted from it.
""")
